<a href="https://colab.research.google.com/github/DaviNegreiros/EyeAiRH/blob/main/C%C3%B3pia_de_Trabalho_2_TEMA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<left><img src="https://i.ibb.co/zWjkHsWJ/marca-final-rgb-campanha-2025-versao02.png" width="35%" height="35%"></left>

# Trabalho 3 — Trabalho Livre — Sintonia & Sincronia
### Disciplina: Tópicos Especiais em Matemática Aplicada: Visão Computacional com Deep Learning 
### Professor: Vinicius Rispoli — FCTE/UnB

---

Autores: Davi Monteiro de Negreiros

Matrícula: 232013971

Data de entrega: 17 de julho de 2026

Local de entrega: https://forms.gle/XXhUrm9rg17a51xh9

---

### Objetivo: 
Que os estudantes apresentem alguma aplicação da visão computacional em qualquer contexto. Deve
ser um trabalho original e não relacionado a qualquer um dos dois trabalhos anteriores. Obs.: não serão aceitos
trabalhos apenas de classificação de imagens, caso seja um problema de classificação deve-se também associar a
segmentação.

**Links anexados:**

Vídeo: https://youtu.be/7iRjcm2KWtM

APK: https://github.com/DaviNegreiros/Sintonia-E-Sincronia/tree/APK

## Sumário

1. Introdução: o que é Visão Computacional e Problema Inicial
2. Media Pipe
3. Processamento de Coreografias em Vídeo
4. Processamento de Poses em Tempo Real
5. Normalização Espacial e Corporal
6. Algoritmo de Comparação de Poses
7. Sistema de Pontuação
8. Sandbox
9. Conclusão
10. Referências

---

## 1. Introdução: O que é Visão Computacional?
Visão Computacional é a ciência e a tecnologia que permitem a interação das máquinas com tudo o que diz respeito ao campo da visão, como imagens e vídeos. Ela está presente em carros autônomos, robôs que fazem entregas, modelos de IA de geração/análise de imagens, robôs industriais, no VAR da Copa do Mundo e até no robô humanoide do Lucas Rangel.

![robôLucasRangel.png](img1.jpg)

As aplicações são muito extensas, portanto, há diversas variações na hora da implementação. No nosso caso, usaremos uma biblioteca chamada MediaPipe, que contém diversas ferramentas úteis para o nosso projeto.

### **Problema Inicial**
No nosso projeto, usaremos Visão Computacional para criar um jogo que lembre vagamente o jogo Just Dance, porém, ao invés de usar sensores complexos e um estúdio enorme para a gravação das coreografias, faremos tudo isso utilizando apenas um celular. Qualquer um poderá subir para o aplicativo qualquer coreografia, em formato de vídeo, de forma offline. Até mesmo a sua própria coreografia. Depois de processado e analisado pelo software utilizando ferramentas do MediaPipe, o vídeo será transformado em um material jogável e, utilizando as mesmas ferramentas do MediaPipe, será possível identificar poses em tempo real do jogador. Essas poses serão comparadas com as poses da coreografia pré-processada em tempo real e trarão feedbacks em relação à qualidade dos movimentos do jogador em tempo real. Ao final da coreografia, o jogador terá acesso aos seus resultados por meio de uma pontuação final.

**Obs:** *Neste notebook, apenas os pontos principais do problema serão apresentados. Porém, um APK que resolve completamente o problema foi desenvolvido pelo aluno e pode ser encontrado no link anexado na seção "Objetivo".*


## 2. MediaPipe

MediaPipe é uma biblioteca do Google para tarefas de visão computacional. No **Sintonia & Sincronia**, usamos o **MediaPipe Tasks Vision** para detectar pose humana em vídeos gravados previamente e na câmera ao vivo.

A dependência usada no APK Android é:
```text
com.google.mediapipe:tasks-vision:0.10.14
```

O modelo configurado no projeto é:
```text
pose_landmarker_full.task
```

O MediaPipe Pose retorna até 33 pontos do corpo, chamados de landmarks. Cada landmark possui `x`, `y`, `z` e `visibility`.
A partir desses pontos essa ferramenta consegue extrair várias informações relevantes como ângulos, posição, distância, tamanhos e etc.

Existem dois modos principais:
- `VIDEO`: usado para processar a coreografia importada;
- `LIVE_STREAM`: usado durante a dança, com a câmera do jogador.

In [25]:
# Setup geral do notebook.

import sys, json, math, time, subprocess, importlib.util, shutil
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    pip_name = pip_name or import_name
    if importlib.util.find_spec(import_name) is not None:
        return True
    try:
        print(f"Instalando {pip_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        return True
    except Exception as exc:
        print(f"Não foi possível instalar {pip_name}: {exc}")
        return False


def resolve_input_path(path):
    """Procura um arquivo no diretório atual, no Colab e no Drive."""
    path = Path(path)
    if path.exists():
        return path
    for root in [Path.cwd(), Path('/content'), Path('/content/drive/MyDrive')]:
        candidate = root / path.name
        if candidate.exists():
            return candidate
    try:
        for candidate in Path.cwd().rglob(path.name):
            if candidate.exists():
                return candidate
    except Exception:
        pass
    return path

HAS_CV2 = ensure_package("cv2", "opencv-python")
HAS_MEDIAPIPE_PACKAGE = ensure_package("mediapipe", "mediapipe")

try:
    import cv2
except Exception as exc:
    cv2 = None; HAS_CV2 = False; print(f"OpenCV indisponível: {exc}")

try:
    import mediapipe as mp
except Exception as exc:
    mp = None; HAS_MEDIAPIPE_PACKAGE = False; print(f"MediaPipe indisponível: {exc}")

try:
    from IPython.display import Video, display
except Exception:
    Video = None; display = None

mp_pose = None
mp_drawing = None
PoseLandmarker = None
PoseLandmarkerOptions = None
BaseOptions = None
RunningMode = None
MP_BACKEND = None
POSE_MODEL_PATH = Path("pose_landmarker_full.task")  #DRIVE#

if HAS_MEDIAPIPE_PACKAGE:
    try:
        from mediapipe.python.solutions import pose as mp_pose
        from mediapipe.python.solutions import drawing_utils as mp_drawing
        MP_BACKEND = "solutions"
    except Exception as solutions_error:
        try:
            from mediapipe.tasks import python as mp_tasks_python
            from mediapipe.tasks.python import vision as mp_tasks_vision
            BaseOptions = mp_tasks_python.BaseOptions
            PoseLandmarker = mp_tasks_vision.PoseLandmarker
            PoseLandmarkerOptions = mp_tasks_vision.PoseLandmarkerOptions
            RunningMode = mp_tasks_vision.RunningMode
            if hasattr(mp, "Image") and hasattr(mp, "ImageFormat"):
                MP_BACKEND = "tasks"
            else:
                print("MediaPipe Tasks foi importado, mas mp.Image/mp.ImageFormat não existem neste ambiente.")
        except Exception as tasks_error:
            print("MediaPipe foi importado, mas não há backend de Pose disponível.")
            print(f"Erro em solutions: {solutions_error}")
            print(f"Erro em tasks: {tasks_error}")

HAS_MEDIAPIPE = MP_BACKEND is not None

if MP_BACKEND == "solutions":
    POSE_CONNECTIONS = list(mp_pose.POSE_CONNECTIONS)
else:
    POSE_CONNECTIONS = [
        (11,13),(13,15),(15,17),(15,19),(15,21),(17,19),
        (12,14),(14,16),(16,18),(16,20),(16,22),(18,20),
        (11,12),(23,24),(11,23),(12,24),
        (23,25),(25,27),(27,29),(27,31),(29,31),
        (24,26),(26,28),(28,30),(28,32),(30,32),
    ]

TARGET_PROCESSING_FPS = 15.0
VISIBILITY_THRESHOLD = 0.35
TIMESTAMP_TOLERANCE_MS = 500.0
Z_DISTANCE_WEIGHT = 0.04
LANDMARK_DISTANCE_THRESHOLD = 1.0

IDX = {
    "left_shoulder": 11, "right_shoulder": 12,
    "left_elbow": 13, "right_elbow": 14,
    "left_wrist": 15, "right_wrist": 16,
    "left_hip": 23, "right_hip": 24,
    "left_knee": 25, "right_knee": 26,
    "left_ankle": 27, "right_ankle": 28,
}

FACE = set(range(11))
HAND_DETAIL = {17,18,19,20,21,22}
FOOT_DETAIL = {29,30,31,32}
TORSO = {11,12,23,24}
PRIMARY = {11,12,13,14,15,16,23,24,25,26,27,28}
MOTION_LANDMARKS = [11,12,13,14,15,16,23,24,25,26,27,28,29,30,31,32]

ANGLE_SPECS = {
    "left_shoulder": (13,11,23), "right_shoulder": (14,12,24),
    "left_elbow": (11,13,15), "right_elbow": (12,14,16),
    "left_hip": (11,23,25), "right_hip": (12,24,26),
    "left_knee": (23,25,27), "right_knee": (24,26,28),
}

SEGMENTS = [(11,13),(13,15),(12,14),(14,16),(23,25),(25,27),(24,26),(26,28),(11,12),(23,24),(11,23),(12,24)]


def r6(x): return round(float(x), 6)


def landmark_component(lm, key, index, default=0.0):
    """Lê landmarks em formato dict ou lista [x, y, z, visibility]."""
    if lm is None:
        return default
    if isinstance(lm, dict):
        return lm.get(key, default)
    if isinstance(lm, (list, tuple)) and len(lm) > index:
        return lm[index]
    return default


def landmark_visibility(lm, default=1.0):
    return float(landmark_component(lm, "visibility", 3, default))


def landmark_x(lm): return float(landmark_component(lm, "x", 0, -1.0))
def landmark_y(lm): return float(landmark_component(lm, "y", 1, -1.0))
def landmark_z(lm): return float(landmark_component(lm, "z", 2, 0.0))
def p3(lm): return (landmark_x(lm), landmark_y(lm), landmark_z(lm))
def sub(a,b): return (a[0]-b[0], a[1]-b[1], a[2]-b[2])
def dot(a,b): return a[0]*b[0] + a[1]*b[1] + a[2]*b[2]
def norm(v): return math.sqrt(dot(v,v))


def raw_landmark_visible(lm, threshold=VISIBILITY_THRESHOLD):
    """Conta apenas landmarks confiáveis e dentro do quadro 0..1."""
    if lm is None:
        return False
    visibility = landmark_visibility(lm, 1.0)
    x, y = landmark_x(lm), landmark_y(lm)
    return visibility >= threshold and 0.0 <= x <= 1.0 and 0.0 <= y <= 1.0


def count_visible_landmarks(landmarks, threshold=VISIBILITY_THRESHOLD):
    return sum(1 for lm in (landmarks or []) if raw_landmark_visible(lm, threshold))


def raw_valid(landmarks, index, threshold=VISIBILITY_THRESHOLD):
    return landmarks is not None and 0 <= index < len(landmarks) and raw_landmark_visible(landmarks[index], threshold)


def valid(landmarks, index, threshold=VISIBILITY_THRESHOLD):
    return landmarks is not None and 0 <= index < len(landmarks) and landmark_visibility(landmarks[index], 1.0) >= threshold


def center(landmarks, a, b):
    pa, pb = p3(landmarks[a]), p3(landmarks[b])
    return ((pa[0]+pb[0])/2, (pa[1]+pb[1])/2, (pa[2]+pb[2])/2)


def normalize_landmarks(landmarks):
    if not landmarks or len(landmarks) <= 24:
        return []
    hip = center(landmarks, 23, 24)
    shoulder = center(landmarks, 11, 12)
    scale = norm(sub(shoulder, hip)) or 1.0
    out = []
    for lm in landmarks:
        x,y,z = p3(lm)
        out.append({"x": r6((x-hip[0])/scale), "y": r6((y-hip[1])/scale), "z": r6((z-hip[2])/scale), "visibility": r6(landmark_visibility(lm, 1.0))})
    return out


def angle(a,b,c):
    v1, v2 = sub(a,b), sub(c,b)
    den = norm(v1) * norm(v2)
    if den <= 1e-8: return 0.0
    return math.degrees(math.acos(max(-1, min(1, dot(v1,v2)/den))))


def calculate_joint_angles(norm_landmarks):
    if not norm_landmarks: return {}
    angles = {}
    for name, (a,b,c) in ANGLE_SPECS.items():
        if max(a,b,c) < len(norm_landmarks):
            angles[name] = r6(angle(p3(norm_landmarks[a]), p3(norm_landmarks[b]), p3(norm_landmarks[c])))
    if len(norm_landmarks) > 24:
        shoulder = center(norm_landmarks, 11, 12)
        hip = center(norm_landmarks, 23, 24)
        angles["torso"] = r6(angle(shoulder, hip, (hip[0], hip[1]-1, hip[2])))
    return angles


def landmark_to_dict(lm):
    visibility = getattr(lm, "visibility", None)
    if visibility is None:
        visibility = getattr(lm, "presence", 1.0)
    return {"x": r6(getattr(lm, "x", 0.0)), "y": r6(getattr(lm, "y", 0.0)), "z": r6(getattr(lm, "z", 0.0)), "visibility": r6(visibility if visibility is not None else 1.0)}


def landmarks_from_result(result):
    if result is None:
        return []
    if MP_BACKEND == "solutions":
        pose_landmarks = getattr(result, "pose_landmarks", None)
        if pose_landmarks is None:
            return []
        return [landmark_to_dict(l) for l in pose_landmarks.landmark]
    poses = getattr(result, "pose_landmarks", None)
    if callable(poses):
        poses = poses()
    if not poses:
        return []
    return [landmark_to_dict(l) for l in poses[0]]


class PoseDetector:
    def __init__(self, backend, detector, running_mode=None):
        self.backend = backend
        self.detector = detector
        self.running_mode = running_mode

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc, tb):
        close = getattr(self.detector, "close", None)
        if callable(close):
            close()


def open_pose(static_image_mode=False):
    if not HAS_MEDIAPIPE:
        raise RuntimeError("MediaPipe não está disponível neste ambiente.")
    if MP_BACKEND == "solutions":
        detector = mp_pose.Pose(static_image_mode=static_image_mode, model_complexity=1, smooth_landmarks=True, enable_segmentation=False, min_detection_confidence=0.5, min_tracking_confidence=0.5)
        return PoseDetector("solutions", detector, "image" if static_image_mode else "video")

    model_path = resolve_input_path(POSE_MODEL_PATH)
    if not model_path.exists():
        raise FileNotFoundError(f"Modelo do Pose Landmarker não encontrado: {model_path} #DRIVE#")
    running_mode = RunningMode.IMAGE if static_image_mode else RunningMode.VIDEO
    options = PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=str(model_path)),
        running_mode=running_mode,
        num_poses=1,
        min_pose_detection_confidence=0.5,
        min_pose_presence_confidence=0.5,
        min_tracking_confidence=0.5,
        output_segmentation_masks=False,
    )
    return PoseDetector("tasks", PoseLandmarker.create_from_options(options), "image" if static_image_mode else "video")


def detect_pose_in_bgr(frame_bgr, pose, timestamp_ms=None):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    if pose.backend == "solutions":
        rgb.flags.writeable = False
        result = pose.detector.process(rgb)
    else:
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        if pose.running_mode == "image":
            result = pose.detector.detect(mp_image)
        else:
            if timestamp_ms is None:
                timestamp_ms = int(time.time() * 1000)
            result = pose.detector.detect_for_video(mp_image, int(timestamp_ms))
    return landmarks_from_result(result), result


def draw_landmark_list_on_frame(frame_bgr, landmarks):
    if not landmarks:
        return frame_bgr
    h, w = frame_bgr.shape[:2]
    for a, b in POSE_CONNECTIONS:
        if raw_valid(landmarks, a, 0.20) and raw_valid(landmarks, b, 0.20):
            ax, ay = int(landmark_x(landmarks[a]) * w), int(landmark_y(landmarks[a]) * h)
            bx, by = int(landmark_x(landmarks[b]) * w), int(landmark_y(landmarks[b]) * h)
            cv2.line(frame_bgr, (ax, ay), (bx, by), (120, 255, 120), 2)
    for lm in landmarks:
        if raw_landmark_visible(lm):
            x, y = int(landmark_x(lm) * w), int(landmark_y(lm) * h)
            cv2.circle(frame_bgr, (x, y), 3, (255, 160, 60), -1)
    return frame_bgr


def draw_landmarks_on_frame(frame_bgr, result=None, landmarks=None):
    if MP_BACKEND == "solutions" and result is not None and getattr(result, "pose_landmarks", None) is not None and mp_drawing is not None:
        mp_drawing.draw_landmarks(frame_bgr, result.pose_landmarks, POSE_CONNECTIONS)
        return frame_bgr
    if landmarks is None:
        landmarks = landmarks_from_result(result)
    return draw_landmark_list_on_frame(frame_bgr, landmarks)


def find_ffmpeg_executable():
    exe = shutil.which("ffmpeg")
    if exe:
        return exe
    if ensure_package("imageio_ffmpeg", "imageio-ffmpeg"):
        try:
            import imageio_ffmpeg
            return imageio_ffmpeg.get_ffmpeg_exe()
        except Exception as exc:
            print(f"imageio-ffmpeg instalado, mas ffmpeg não pôde ser carregado: {exc}")
    return None


def make_browser_compatible_video(input_path, output_path):
    """Reencoda o vídeo de debug para H.264/yuv420p, formato aceito por VS Code e notebooks."""
    input_path, output_path = Path(input_path), Path(output_path)
    if not input_path.exists():
        return output_path
    exe = find_ffmpeg_executable()
    if exe is None:
        print("ffmpeg não está disponível; mantendo o MP4 bruto do OpenCV, que pode não tocar no VS Code/notebook.")
        if input_path != output_path:
            input_path.replace(output_path)
        return output_path
    tmp_output = output_path.with_name(output_path.stem + "_h264.mp4")
    cmd = [exe, "-y", "-i", str(input_path), "-an", "-c:v", "libx264", "-pix_fmt", "yuv420p", "-movflags", "+faststart", str(tmp_output)]
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if result.returncode != 0 or not tmp_output.exists():
        print("Não foi possível converter o vídeo para H.264. Mantendo o MP4 bruto do OpenCV.")
        print(result.stderr[-800:])
        if input_path != output_path:
            input_path.replace(output_path)
        return output_path
    tmp_output.replace(output_path)
    try:
        if input_path.exists() and input_path != output_path:
            input_path.unlink()
    except Exception:
        pass
    print(f"Vídeo compatível com VS Code/notebook salvo em: {output_path}")
    return output_path


def display_video_if_exists(path):
    path = Path(path)
    if path.exists() and Video is not None and display is not None:
        display(Video(str(path), embed=False))
    else:
        print(f"Vídeo não encontrado ou exibição indisponível: {path}")

print(f"OpenCV disponível: {HAS_CV2}")
print(f"MediaPipe disponível: {HAS_MEDIAPIPE}")
print(f"Backend MediaPipe: {MP_BACKEND or 'nenhum'}")
if MP_BACKEND == "tasks":
    print(f"Modelo usado pelo Tasks: {resolve_input_path(POSE_MODEL_PATH)}")


OpenCV disponível: True
MediaPipe disponível: True
Backend MediaPipe: tasks
Modelo usado pelo Tasks: pose_landmarker_full.task


## 3. Processamento de Coreografias em Vídeo

Aqui ocorrerá o processo de transformar um vídeo em uma coreografia jogável.
O vídeo da coreografia importada vira um **moveset**.

No aplicativo Android, o fluxo é:
```text
vídeo -> decoder -> frame RGBA -> MediaPipe -> landmarks -> normalização -> ângulos -> moveset.json
```

O decoder processa o vídeo em **15 FPS**, para reduzir custo e manter timestamps estáveis.

Para cada frame processado, salvamos:

- número do frame;
- timestamp;
- se uma pose foi detectada;
- landmarks originais;
- landmarks normalizados;
- ângulos principais do corpo.

Também é gerado um vídeo de debug com o esqueleto do MediaPipe Pose desenhado por cima.

Arquivos usados nesta seção:

```text
dance2.mp4       #DRIVE#
dance2_pose.mp4  #DRIVE#
```

In [26]:
VIDEO_PATH = Path("dance2.mp4")             #DRIVE#
DEBUG_VIDEO_PATH = Path("dance2_pose.mp4") #DRIVE#
MOVESET_PATH = Path("moveset_dance2.json") #DRIVE#
EXISTING_MOVESET_FALLBACK = Path("dance2_moveset.json") #DRIVE#


def should_process_frame(timestamp_ms, next_sample_ms):
    """Decide se o frame pertence à linha do tempo amostrada."""
    return timestamp_ms + 0.001 >= next_sample_ms


def generate_moveset(video_path=VIDEO_PATH, moveset_path=MOVESET_PATH, debug_video_path=DEBUG_VIDEO_PATH, target_fps=TARGET_PROCESSING_FPS):
    video_path = resolve_input_path(video_path)
    moveset_path, debug_video_path = Path(moveset_path), Path(debug_video_path)
    if not HAS_CV2 or not HAS_MEDIAPIPE:
        print("OpenCV/MediaPipe não estão disponíveis. A geração foi pulada.")
        return None
    if not video_path.exists():
        print(f"Arquivo não encontrado: {video_path} #DRIVE#")
        return None

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"Não foi possível abrir o vídeo: {video_path}")
        return None

    source_fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0)
    if source_fps <= 0:
        source_fps = float(target_fps)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    source_frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    duration = source_frame_count / source_fps if source_fps > 0 else 0.0

    effective_processing_fps = max(0.001, min(float(target_fps), source_fps))
    interval_ms = 1000.0 / effective_processing_fps

    writer = None
    raw_debug_video_path = debug_video_path.with_name(debug_video_path.stem + "_raw.mp4")
    if width > 0 and height > 0:
        writer = cv2.VideoWriter(
            str(raw_debug_video_path),
            cv2.VideoWriter_fourcc(*"mp4v"),
            effective_processing_fps,
            (width, height),
        )
        if not writer.isOpened():
            print("Não foi possível criar o vídeo de debug com OpenCV.")
            writer = None

    frames, next_sample_ms = [], 0.0
    frame_index = 0
    processed_count = 0
    last_detector_ms = -1

    with open_pose(static_image_mode=False) as pose:
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            
            timestamp_ms = frame_index * 1000.0 / source_fps
            if not should_process_frame(timestamp_ms, next_sample_ms):
                frame_index += 1
                continue
            while next_sample_ms <= timestamp_ms + 0.001:
                next_sample_ms += interval_ms

            detector_ms = max(int(round(timestamp_ms)), last_detector_ms + 1)
            last_detector_ms = detector_ms
            landmarks, result = detect_pose_in_bgr(frame, pose, detector_ms)
            visible_count = count_visible_landmarks(landmarks)
            normalized = normalize_landmarks(landmarks) if landmarks else []
            angles = calculate_joint_angles(normalized) if normalized else {}
            frames.append({
                "frame": frame_index,
                "timestamp": r6(timestamp_ms / 1000.0),
                "pose_detected": visible_count > 0,
                "visible_landmarks": visible_count,
                "landmarks": landmarks,
                "normalized_landmarks": normalized,
                "joint_angles": angles,
            })

            if writer is not None:
                writer.write(draw_landmarks_on_frame(frame.copy(), result, landmarks))
            frame_index += 1
            processed_count += 1
            if processed_count % 50 == 0:
                print(f"Processados {processed_count} frames...")

    cap.release()
    if writer is not None:
        writer.release()
        make_browser_compatible_video(raw_debug_video_path, debug_video_path)

    actual_processing_fps = (len(frames) / duration) if duration > 0 else effective_processing_fps
    moveset = {
        "metadata": {
            "video": video_path.name,
            "fps": source_fps,
            "target_processing_fps": float(target_fps),
            "processing_fps": r6(actual_processing_fps),
            "debug_video_fps": r6(effective_processing_fps),
            "source_frame_count": source_frame_count,
            "frame_count": len(frames),
            "width": width,
            "height": height,
            "duration": r6(duration),
        },
        "frames": frames,
    }
    moveset_path.write_text(json.dumps(moveset, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Moveset salvo em: {moveset_path}")
    print(f"Vídeo de debug salvo em: {debug_video_path}")
    print(f"Original: {source_frame_count} frames, {source_fps:.3f} FPS, {duration:.3f}s")
    print(f"Processado: {len(frames)} frames, FPS efetivo {actual_processing_fps:.3f}")
    visible_counts = [f.get("visible_landmarks", 0) for f in frames]
    if visible_counts:
        avg_visible = sum(visible_counts) / len(visible_counts)
        print(f"Landmarks visíveis por frame: média {avg_visible:.1f}, mínimo {min(visible_counts)}, máximo {max(visible_counts)}")
    return moveset


moveset_data = generate_moveset()

if moveset_data is None and resolve_input_path(EXISTING_MOVESET_FALLBACK).exists():
    fallback = resolve_input_path(EXISTING_MOVESET_FALLBACK)
    print(f"Usando moveset existente como fallback didático: {fallback} #DRIVE#")
    moveset_data = json.loads(fallback.read_text(encoding="utf-8"))

if resolve_input_path(VIDEO_PATH).exists():
    print("Vídeo original:")
    display_video_if_exists(resolve_input_path(VIDEO_PATH))
if DEBUG_VIDEO_PATH.exists():
    print("Vídeo processado com MediaPipe:")
    display_video_if_exists(DEBUG_VIDEO_PATH)


Processados 50 frames...
Processados 100 frames...
Processados 150 frames...
Processados 200 frames...
Vídeo compatível com VS Code/notebook salvo em: dance2_pose.mp4
Moveset salvo em: moveset_dance2.json
Vídeo de debug salvo em: dance2_pose.mp4
Original: 400 frames, 30.000 FPS, 13.333s
Processado: 200 frames, FPS efetivo 15.000
Landmarks visíveis por frame: média 33.0, mínimo 33, máximo 33
Vídeo original:


Vídeo processado com MediaPipe:


## 4. Processamento de Poses em Tempo Real

Nesta seção ficam as estruturas usadas para transformar frames da câmera em poses analisáveis em tempo real.

No Sandbox, essas funções são usadas para abrir a webcam e comparar com a coreografia.


In [27]:
from dataclasses import dataclass, field
from types import MappingProxyType
from typing import Mapping


@dataclass(frozen=True, slots=True)
class PoseFrame:
    """Dados de pose de um único frame.

    O timestamp é sempre expresso em milissegundos, como nos notebooks originais.
    """

    frame: int
    timestamp: float
    pose_detected: bool
    landmarks: list[dict[str, float]]
    normalized_landmarks: list[dict[str, float]]
    joint_angles: Mapping[str, float] = field(default_factory=dict)

    def __post_init__(self):
        object.__setattr__(self, "joint_angles", MappingProxyType(dict(self.joint_angles)))


class PoseSkeletonDrawer:
    """Desenha o esqueleto do MediaPipe usando as conexões oficiais."""

    @staticmethod
    def draw(frame, landmarks):
        return draw_landmark_list_on_frame(frame, landmarks)


def create_pose_frame(frame_index, timestamp_ms, landmarks):
    """Cria PoseFrame a partir dos landmarks crus retornados pelo MediaPipe."""
    visible_count = count_visible_landmarks(landmarks)
    normalized = normalize_landmarks(landmarks) if landmarks else []
    angles = calculate_joint_angles(normalized) if normalized else {}
    return PoseFrame(
        frame=int(frame_index),
        timestamp=float(timestamp_ms),
        pose_detected=visible_count > 0,
        landmarks=landmarks or [],
        normalized_landmarks=normalized,
        joint_angles=angles,
    )


def pose_frame_from_moveset_item(item):
    """Converte um item do moveset JSON para PoseFrame."""
    landmarks = item.get("landmarks", []) or []
    normalized = item.get("normalized_landmarks", []) or []
    if landmarks and not normalized:
        normalized = normalize_landmarks(landmarks)
    angles = item.get("joint_angles", {}) or {}
    if normalized and not angles:
        angles = calculate_joint_angles(normalized)
    timestamp = float(item.get("timestamp", 0.0)) * 1000.0
    return PoseFrame(
        frame=int(item.get("frame", 0)),
        timestamp=timestamp,
        pose_detected=count_visible_landmarks(landmarks) > 0,
        landmarks=landmarks,
        normalized_landmarks=normalized,
        joint_angles={str(key): float(value) for key, value in angles.items()},
    )


def run_webcam_pose_preview(camera_index=0, mirror=True):
    """Preview simples do Notebook 02: webcam + esqueleto. Não é executado automaticamente."""
    if not HAS_CV2 or not HAS_MEDIAPIPE:
        print("OpenCV/MediaPipe não estão disponíveis neste ambiente.")
        return

    cap = cv2.VideoCapture(camera_index)
    if not cap.isOpened():
        print("Não foi possível abrir a câmera. Em Colab isso é esperado sem integração via navegador.")
        return

    frame_index = 0
    last_detector_ms = -1
    start = time.time()
    drawer = PoseSkeletonDrawer()
    with open_pose(static_image_mode=False) as pose:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if mirror:
                frame = cv2.flip(frame, 1)
            timestamp_ms = max(int((time.time() - start) * 1000), last_detector_ms + 1)
            last_detector_ms = timestamp_ms
            landmarks, _ = detect_pose_in_bgr(frame, pose, timestamp_ms)
            pose_frame = create_pose_frame(frame_index, timestamp_ms, landmarks)
            if pose_frame.pose_detected:
                drawer.draw(frame, pose_frame.landmarks)
            cv2.putText(frame, f"Landmarks visiveis: {count_visible_landmarks(landmarks)}", (16,32), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)
            cv2.imshow("Pose em tempo real - pressione Q para sair", frame)
            frame_index += 1
            key = cv2.waitKey(1) & 0xFF
            if key in (ord("q"), ord("Q"), 27):
                break

    cap.release()
    cv2.destroyAllWindows()


print("Funções de pose em tempo real carregadas. Execute run_webcam_pose_preview() apenas se quiser abrir a câmera.")


Funções de pose em tempo real carregadas. Execute run_webcam_pose_preview() apenas se quiser abrir a câmera.


## 5. Normalização Espacial e Corporal

- As pessoas podem aparecer em posições diferentes na câmera por diversos motivos.
- As pessoas possuem corpos de tamanhos, medidas e ângulos diferentes.



Por isso, antes de comparar duas poses, normalizaremos os landmarks.<br>A normalização usada no projeto faz duas coisas:

1. move o centro do quadril para a origem;
2. divide todos os pontos pela distância entre o centro do quadril e o centro dos ombros.

Assim, a comparação fica menos sensível à distância da câmera e à posição da pessoa na tela. Depois disso, calculamos ângulos como cotovelo, ombro, quadril, joelho e tronco.

In [28]:
NORMALIZED_MOVESET_PATH = Path("moveset_dance2_normalized.json")


def normalize_moveset_file(input_path=MOVESET_PATH, output_path=NORMALIZED_MOVESET_PATH):
    """Recalcula landmarks normalizados e ângulos de um moveset."""
    input_path, output_path = Path(input_path), Path(output_path)
    if not input_path.exists():
        fallback = resolve_input_path(EXISTING_MOVESET_FALLBACK)
        if fallback.exists():
            print(f"Moveset principal não encontrado. Usando fallback: {fallback} #DRIVE#")
            input_path = fallback
    if not input_path.exists():
        print(f"Moveset não encontrado: {input_path}")
        return None

    data = json.loads(input_path.read_text(encoding="utf-8"))
    normalized_count = 0
    for frame in data.get("frames", []):
        landmarks = frame.get("landmarks", [])
        visible_count = count_visible_landmarks(landmarks)
        normalized = normalize_landmarks(landmarks) if landmarks else []
        frame["pose_detected"] = visible_count > 0
        frame["visible_landmarks"] = visible_count
        frame["normalized_landmarks"] = normalized
        frame["joint_angles"] = calculate_joint_angles(normalized) if normalized else {}
        if normalized:
            normalized_count += 1

    output_path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Moveset normalizado salvo em: {output_path}")
    print(f"Frames com pose normalizada: {normalized_count}/{len(data.get('frames', []))}")
    return data

normalized_moveset_data = normalize_moveset_file()


Moveset normalizado salvo em: moveset_dance2_normalized.json
Frames com pose normalizada: 200/200


## 6. Algoritmo de Comparação de Poses

Esta seção contém o algorítmo de comparação entre moveset e poses em tempo real.

- busca frames do moveset próximos ao timestamp atual;
- compara landmarks normalizados;
- compara ângulos principais;
- escolhe a melhor correspondência dentro da tolerância temporal;
- mantém média móvel de score;
- emite feedbacks periódicos: `Eita!`, `Ok`, `Bom`, `Ótimo!` e `Perfeito!`.

In [29]:
import bisect
from collections import deque
from dataclasses import dataclass, replace


TIMESTAMP_TOLERANCE_MS_NOTEBOOK = 200.0
FEEDBACK_INTERVAL_MS_NOTEBOOK = 5000.0
SLIDING_WINDOW_MS_NOTEBOOK = 5000.0
LANDMARK_SIMILARITY_WEIGHT_NOTEBOOK = 0.70
ANGLE_SIMILARITY_WEIGHT_NOTEBOOK = 0.30


@dataclass(frozen=True, slots=True)
class FrameEvaluation:
    """Resultado de uma comparação em um instante."""

    timestamp: float
    overall_similarity: float
    landmark_similarity: float
    angle_similarity: float
    confidence: float
    feedback: str = ""
    reference_frame: int | None = None
    reference_timestamp: float | None = None


@dataclass(frozen=True, slots=True)
class GameResult:
    """Resultado consolidado de uma dança completa."""

    final_score: float
    ranque: str
    similarity_average: float
    feedback_score: float


class ReferenceTimeline:
    """Linha do tempo do moveset com busca por timestamp."""

    def __init__(self, frames):
        self.frames = tuple(sorted(frames, key=lambda pose_frame: pose_frame.timestamp))
        self.timestamps = [pose_frame.timestamp for pose_frame in self.frames]

    def get_poses_near(self, timestamp_ms, tolerance_ms):
        start = timestamp_ms - tolerance_ms
        end = timestamp_ms + tolerance_ms
        start_index = bisect.bisect_left(self.timestamps, start)
        end_index = bisect.bisect_right(self.timestamps, end)
        return list(self.frames[start_index:end_index])


class PoseComparator:
    """Comparador híbrido do Notebook 03: landmarks normalizados + ângulos."""

    def __init__(self, landmark_distance_threshold=LANDMARK_DISTANCE_THRESHOLD, landmark_weight=LANDMARK_SIMILARITY_WEIGHT_NOTEBOOK, angle_weight=ANGLE_SIMILARITY_WEIGHT_NOTEBOOK):
        self.landmark_distance_threshold = landmark_distance_threshold
        self.landmark_weight = landmark_weight
        self.angle_weight = angle_weight

    def compare(self, reference_pose, realtime_pose):
        if not reference_pose.pose_detected or not realtime_pose.pose_detected:
            return FrameEvaluation(realtime_pose.timestamp, 0.0, 0.0, 0.0, 0.0, reference_frame=reference_pose.frame, reference_timestamp=reference_pose.timestamp)

        landmark_similarity = self._landmark_similarity(reference_pose, realtime_pose)
        angle_similarity = self._angle_similarity(reference_pose, realtime_pose)
        confidence = self._confidence(reference_pose, realtime_pose)
        overall = (self.landmark_weight * landmark_similarity) + (self.angle_weight * angle_similarity)
        return FrameEvaluation(
            timestamp=realtime_pose.timestamp,
            overall_similarity=max(0.0, min(1.0, overall)),
            landmark_similarity=landmark_similarity,
            angle_similarity=angle_similarity,
            confidence=confidence,
            reference_frame=reference_pose.frame,
            reference_timestamp=reference_pose.timestamp,
        )

    def _landmark_similarity(self, reference_pose, realtime_pose):
        reference = reference_pose.normalized_landmarks
        realtime = realtime_pose.normalized_landmarks
        if not reference or not realtime:
            return 0.0

        total, count = 0.0, 0
        max_len = min(len(reference), len(realtime))
        for index in range(max_len):
            if not valid(reference, index):
                continue
            if not valid(realtime, index):
                distance = self.landmark_distance_threshold
            else:
                ref = reference[index]
                player = realtime[index]
                dx = landmark_x(ref) - landmark_x(player)
                dy = landmark_y(ref) - landmark_y(player)
                dz = landmark_z(ref) - landmark_z(player)
                distance = math.sqrt(dx*dx + dy*dy + dz*dz)
            total += max(0.0, min(1.0, 1.0 - distance / self.landmark_distance_threshold))
            count += 1
        return total / count if count else 0.0

    @staticmethod
    def _angle_similarity(reference_pose, realtime_pose):
        common_keys = set(reference_pose.joint_angles).intersection(realtime_pose.joint_angles)
        if not common_keys:
            return 0.0
        differences = [abs(float(reference_pose.joint_angles[key]) - float(realtime_pose.joint_angles[key])) for key in common_keys]
        mean_difference = sum(differences) / len(differences)
        return max(0.0, min(1.0, 1.0 - mean_difference / 180.0))

    @staticmethod
    def _confidence(reference_pose, realtime_pose):
        ref_values = [landmark_visibility(lm, 0.0) for lm in reference_pose.landmarks if raw_landmark_visible(lm)]
        player_values = [landmark_visibility(lm, 0.0) for lm in realtime_pose.landmarks if raw_landmark_visible(lm)]
        if not ref_values or not player_values:
            return 0.0
        return max(0.0, min(1.0, min(sum(ref_values)/len(ref_values), sum(player_values)/len(player_values))))


class FeedbackMapper:
    """Mapeia score 0-100 para feedback textual e cor."""

    COLORS = {
        "Eita!": (40, 40, 230),
        "Ok": (40, 150, 255),
        "Bom": (40, 220, 255),
        "Ótimo!": (60, 220, 80),
        "Perfeito!": (255, 160, 40),
    }

    def map_score(self, score):
        if score < 25.0:
            return "Eita!"
        if score < 50.0:
            return "Ok"
        if score < 70.0:
            return "Bom"
        if score < 85.0:
            return "Ótimo!"
        return "Perfeito!"

    def color_for(self, feedback):
        return self.COLORS.get(feedback, (255, 255, 255))


class ComparisonEngine:
    """Coordena comparação temporal, média móvel e feedbacks periódicos."""

    def __init__(self, moveset_timeline, comparator=None, feedback_mapper=None, timestamp_tolerance_ms=TIMESTAMP_TOLERANCE_MS_NOTEBOOK, feedback_interval_ms=FEEDBACK_INTERVAL_MS_NOTEBOOK, sliding_window_ms=SLIDING_WINDOW_MS_NOTEBOOK):
        self.moveset_timeline = moveset_timeline
        self.comparator = comparator or PoseComparator()
        self.feedback_mapper = feedback_mapper or FeedbackMapper()
        self.timestamp_tolerance_ms = timestamp_tolerance_ms
        self.feedback_interval_ms = feedback_interval_ms
        self.sliding_window_ms = sliding_window_ms
        self.score_window = deque()
        self.all_scores = []
        self.feedback_history = []
        self.current_feedback = "Eita!"
        self._last_feedback_timestamp = 0.0

    def compare(self, realtime_pose):
        candidates = self.moveset_timeline.get_poses_near(realtime_pose.timestamp, self.timestamp_tolerance_ms)
        if not candidates:
            result = FrameEvaluation(realtime_pose.timestamp, 0.0, 0.0, 0.0, 0.0, self.current_feedback)
            self._update_score_window(result)
            return replace(result, feedback=self.current_feedback)

        result = max((self.comparator.compare(reference_pose, realtime_pose) for reference_pose in candidates), key=lambda item: item.overall_similarity)
        self._update_score_window(result)
        return replace(result, feedback=self.current_feedback)

    def current_score(self):
        if not self.score_window:
            return 0.0
        return sum(score for _, score in self.score_window) / len(self.score_window)

    def similarity_average(self):
        if not self.all_scores:
            return 0.0
        return sum(self.all_scores) / len(self.all_scores)

    def _update_score_window(self, result):
        score = result.overall_similarity * 100.0
        self.all_scores.append(score)
        self.score_window.append((result.timestamp, score))

        min_timestamp = result.timestamp - self.sliding_window_ms
        while self.score_window and self.score_window[0][0] < min_timestamp:
            self.score_window.popleft()

        if result.timestamp - self._last_feedback_timestamp >= self.feedback_interval_ms:
            self.current_feedback = self.feedback_mapper.map_score(self.current_score())
            self.feedback_history.append(self.current_feedback)
            self._last_feedback_timestamp = result.timestamp


class FinalScoreCalculator:
    """Calcula nota final combinando similaridade contínua e feedbacks."""

    FEEDBACK_POINTS = {
        "Eita!": 1,
        "Ok": 2,
        "Bom": 3,
        "Ótimo!": 4,
        "Perfeito!": 5,
    }
    MAX_FEEDBACK_POINTS = 5

    def calculate(self, feedbacks, similarity_average, duration_seconds):
        if duration_seconds < 0:
            raise ValueError("A duração da música não pode ser negativa.")
        bounded_similarity = max(0.0, min(100.0, float(similarity_average)))
        feedback_score = self._feedback_score(feedbacks)
        final_score = max(0.0, min(100.0, bounded_similarity * 0.70 + feedback_score * 0.30))
        return GameResult(final_score, self._rank(final_score), bounded_similarity, feedback_score)

    def _feedback_score(self, feedbacks):
        if not feedbacks:
            return 0.0
        total_points = sum(self.FEEDBACK_POINTS.get(feedback, 0) for feedback in feedbacks)
        max_points = len(feedbacks) * self.MAX_FEEDBACK_POINTS
        return max(0.0, min(100.0, (total_points / max_points) * 100.0))

    @staticmethod
    def _rank(score):
        if score >= 95.0:
            return "S"
        if score >= 90.0:
            return "A+"
        if score >= 80.0:
            return "A"
        if score >= 70.0:
            return "B"
        if score >= 60.0:
            return "C"
        if score >= 40.0:
            return "D"
        return "E"


def load_moveset_timeline(path):
    """Carrega um moveset JSON e devolve a timeline pronta para comparação."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Moveset não encontrado: {path}")
    data = json.loads(path.read_text(encoding="utf-8"))
    frames = [pose_frame_from_moveset_item(item) for item in data.get("frames", [])]
    return ReferenceTimeline(frames)


print("Motor de comparação carregado. Esta célula não abre câmera nem executa o jogo.")


Motor de comparação carregado. Esta célula não abre câmera nem executa o jogo.


## 7. Sandbox

Aqui será possível realmente ter um gostinho de como o aplicativo funciona. Todas as seções acima se encontram nesse ponto para que um pequeno jogo possa acontecer.

```text
macareninha.mp4           #DRIVE#
macareninha_pose.mp4      #DRIVE#
macareninha_moveset.json  #DRIVE#
```

Essa seção processa `macareninha.mp4`, gera o moveset e o vídeo com pose, e depois inicia o jogo.

Aqui ocorre um pequeno countdown, o vídeo processado é apresentado à esquerda e o a camêra do jogador à direita. Ambos com os landmarks desenhados sobre o corpo para permitir a comparação.

In [32]:
SANDBOX_CONFIG = {
    "raw_video_path": Path("macareninha.mp4"),            #DRIVE#
    "reference_video_path": Path("macareninha_pose.mp4"), #DRIVE#
    "moveset_path": Path("macareninha_moveset.json"),     #DRIVE#
    "camera_index": 0,
    "countdown_labels": ("5", "4", "3", "2", "1", "JÁ!"),
    "mirror_camera": True,
    "force_process_video": True,
}


class VideoPlayer:
    """Player OpenCV sincronizado por tempo, inspirado no Notebook 03."""

    def __init__(self, video_path):
        self.video_path = Path(video_path)
        self.capture = None
        self.fps = 0.0
        self.frame_count = 0
        self.current_frame_index = 0
        self.current_frame = None
        self._started_at = None
        self._paused = True

    def open(self):
        if not self.video_path.exists():
            raise FileNotFoundError(f"Vídeo não encontrado: {self.video_path}")
        self.capture = cv2.VideoCapture(str(self.video_path))
        if not self.capture.isOpened():
            raise RuntimeError(f"Não foi possível abrir o vídeo: {self.video_path}")
        self.fps = float(self.capture.get(cv2.CAP_PROP_FPS) or 0.0)
        self.frame_count = int(self.capture.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
        if self.fps <= 0 or self.frame_count <= 0:
            raise ValueError("Metadados inválidos no vídeo de referência.")
        success, frame = self.capture.read()
        if not success or frame is None:
            raise RuntimeError("Não foi possível ler o primeiro frame do vídeo.")
        self.current_frame = frame
        self.current_frame_index = 0
        self.capture.set(cv2.CAP_PROP_POS_FRAMES, 0)

    def play(self):
        self._started_at = time.perf_counter() - (self.timestamp_ms / 1000.0)
        self._paused = False

    def pause(self):
        self._paused = True

    def reset(self):
        if self.capture is None:
            return
        self.capture.set(cv2.CAP_PROP_POS_FRAMES, 0)
        success, frame = self.capture.read()
        if success and frame is not None:
            self.current_frame = frame
        self.capture.set(cv2.CAP_PROP_POS_FRAMES, 0)
        self.current_frame_index = 0
        self._started_at = None
        self._paused = True

    def read(self):
        if self.capture is None:
            raise RuntimeError("VideoPlayer não foi inicializado.")
        if self._paused:
            return True, self.current_frame
        target_frame = min(int((self.timestamp_ms / 1000.0) * self.fps), self.frame_count - 1)
        if target_frame != self.current_frame_index:
            self.capture.set(cv2.CAP_PROP_POS_FRAMES, target_frame)
            success, frame = self.capture.read()
            if not success or frame is None:
                self.current_frame_index = self.frame_count - 1
                return self.current_frame is not None, self.current_frame
            self.current_frame = frame
            self.current_frame_index = target_frame
        return True, self.current_frame

    @property
    def timestamp_ms(self):
        if self._started_at is None:
            return (self.current_frame_index / self.fps) * 1000.0 if self.fps > 0 else 0.0
        return max(0.0, (time.perf_counter() - self._started_at) * 1000.0)

    @property
    def duration_ms(self):
        return (self.frame_count / self.fps) * 1000.0 if self.fps > 0 else 0.0

    @property
    def is_finished(self):
        return self.timestamp_ms >= self.duration_ms if self._started_at is not None else self.current_frame_index >= self.frame_count - 1

    def release(self):
        if self.capture is not None:
            self.capture.release()
            self.capture = None


class ComparisonHUD:
    """HUD colorido do jogo, baseado no Notebook 03."""

    window_name = "Dance Comparison"

    def __init__(self, panel_width=640, panel_height=480):
        self.panel_width = panel_width
        self.panel_height = panel_height
        self.feedback_mapper = FeedbackMapper()
        self.drawer = PoseSkeletonDrawer()

    def draw(self, reference_frame, webcam_frame, realtime_pose, result, fps, score, feedback, countdown_text=None):
        reference = self._resize(reference_frame)
        webcam = self._resize(webcam_frame)
        if realtime_pose.pose_detected:
            self.drawer.draw(webcam, realtime_pose.landmarks)
        canvas = cv2.hconcat([reference, webcam])
        self._draw_hud(canvas, result, fps, score, feedback, countdown_text)
        return canvas

    def show(self, image):
        cv2.imshow(self.window_name, image)

    def draw_countdown(self, reference_frame, webcam_frame, text, fps=0.0):
        empty_pose = PoseFrame(0, 0.0, False, [], [], {})
        empty_result = FrameEvaluation(0.0, 0.0, 0.0, 0.0, 0.0, "")
        return self.draw(reference_frame, webcam_frame, empty_pose, empty_result, fps, 0.0, "", text)

    def draw_final_result(self, result):
        screen = cv2.UMat(520, 760, cv2.CV_8UC3).get()
        screen[:] = (18, 18, 18)
        self._draw_centered_text(screen, "FIM DA DANÇA", 95, 1.6, (0, 255, 255), 3)
        self._draw_centered_text(screen, "Pontuação Final", 180, 1.1, (240, 240, 240), 2)
        self._draw_centered_text(screen, f"{result.final_score:.1f} / 100", 250, 1.45, (255, 255, 255), 3)
        self._draw_centered_text(screen, f"Rank: {result.ranque}", 325, 1.35, (0, 255, 255), 3)
        self._draw_centered_text(screen, f"Similaridade: {result.similarity_average:.1f}", 395, 0.8, (210, 210, 210), 2)
        self._draw_centered_text(screen, "Pressione Q para fechar", 465, 0.75, (180, 180, 180), 2)
        return screen

    def _draw_hud(self, canvas, result, fps, score, feedback, countdown_text):
        if countdown_text:
            self._draw_center_text(canvas, countdown_text)
        feedback_color = self.feedback_mapper.color_for(feedback)
        lines = (
            f"Tempo: {result.timestamp / 1000.0:05.1f}s",
            f"FPS: {fps:05.1f}",
            f"Score Atual: {score:05.1f}",
            f"Feedback: {feedback}",
        )
        x, y = 18, 34
        for index, line in enumerate(lines):
            color = feedback_color if line.startswith("Feedback") else (255, 255, 255)
            y_position = y + index * 28
            cv2.putText(canvas, line, (x, y_position), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 4, cv2.LINE_AA)
            cv2.putText(canvas, line, (x, y_position), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 1, cv2.LINE_AA)
        cv2.putText(canvas, "Referencia", (18, canvas.shape[0] - 18), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.putText(canvas, "Webcam", (self.panel_width + 18, canvas.shape[0] - 18), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    def _draw_center_text(self, canvas, text):
        font = cv2.FONT_HERSHEY_SIMPLEX
        scale = 3.0 if len(text) <= 3 else 2.0
        thickness = 7
        size, _ = cv2.getTextSize(text, font, scale, thickness)
        x = (canvas.shape[1] - size[0]) // 2
        y = (canvas.shape[0] + size[1]) // 2
        cv2.putText(canvas, text, (x, y), font, scale, (0, 0, 0), thickness + 4, cv2.LINE_AA)
        cv2.putText(canvas, text, (x, y), font, scale, (0, 255, 255), thickness, cv2.LINE_AA)

    @staticmethod
    def _draw_centered_text(image, text, y, scale, color, thickness):
        font = cv2.FONT_HERSHEY_SIMPLEX
        size, _ = cv2.getTextSize(text, font, scale, thickness)
        x = (image.shape[1] - size[0]) // 2
        cv2.putText(image, text, (x, y), font, scale, color, thickness, cv2.LINE_AA)

    def _resize(self, frame):
        return cv2.resize(frame, (self.panel_width, self.panel_height), interpolation=cv2.INTER_AREA)


class DanceComparisonSandbox:
    """Fluxo jogável isolado no Sandbox."""

    def __init__(self, config=SANDBOX_CONFIG):
        self.config = config
        self.moveset_path = Path(config["moveset_path"])
        self.video_path = Path(config["reference_video_path"])
        self.raw_video_path = Path(config["raw_video_path"])
        self.camera_index = int(config["camera_index"])
        self.countdown_labels = tuple(config["countdown_labels"])
        self.mirror_camera = bool(config["mirror_camera"])
        self.camera = None
        self.video_player = None
        self.pose_context = None
        self.timeline = None
        self.engine = None
        self.hud = ComparisonHUD()
        self.final_score_calculator = FinalScoreCalculator()
        self.fps = 0.0
        self._last_tick = time.perf_counter()
        self._last_detector_ms = -1

    def prepare_assets(self):
        if not self.raw_video_path.exists():
            raise FileNotFoundError(f"Arquivo de vídeo macareninha não encontrado: {self.raw_video_path} #DRIVE#")
        force = bool(self.config.get("force_process_video", True))
        if not force and self.moveset_path.exists() and self.video_path.exists():
            return
        print("Processando macareninha.mp4 para gerar macareninha_moveset.json e macareninha_pose.mp4...")
        generate_moveset(
            video_path=self.raw_video_path,
            moveset_path=self.moveset_path,
            debug_video_path=self.video_path,
            target_fps=TARGET_PROCESSING_FPS,
        )

    def run(self):
        self.prepare_assets()
        if not HAS_CV2 or not HAS_MEDIAPIPE:
            print("OpenCV/MediaPipe não estão disponíveis neste ambiente.")
            return None
        self.timeline = load_moveset_timeline(self.moveset_path)
        self.engine = ComparisonEngine(self.timeline, PoseComparator(), FeedbackMapper())
        self.video_player = VideoPlayer(self.video_path)
        self.camera = cv2.VideoCapture(self.camera_index)
        if not self.camera.isOpened():
            print("Não foi possível abrir a câmera. Em Colab isso é esperado.")
            self.camera.release()
            return None
        self.video_player.open()
        self.video_player.pause()

        try:
            with open_pose(static_image_mode=False) as pose:
                self.pose_context = pose
                if self._run_countdown():
                    self.video_player.reset()
                    self.video_player.play()
                    completed = self._run_game_loop()
                    if completed:
                        result = self._compute_final_result()
                        self._show_final_result(result)
                        return result
        finally:
            self._release_resources()
        return None

    def _run_countdown(self):
        for text in self.countdown_labels:
            started = time.perf_counter()
            duration = 0.75 if text == "JÁ!" else 1.0
            while time.perf_counter() - started < duration:
                ok_cam, webcam_frame = self.camera.read()
                ok_ref, reference_frame = self.video_player.read()
                if not ok_cam or webcam_frame is None or not ok_ref or reference_frame is None:
                    raise RuntimeError("Não foi possível renderizar o countdown.")
                if self.mirror_camera:
                    webcam_frame = cv2.flip(webcam_frame, 1)
                view = self.hud.draw_countdown(reference_frame, webcam_frame, text, self.fps)
                self.hud.show(view)
                if self._should_quit():
                    return False
        return True

    def _run_game_loop(self):
        frame_index = 0
        while True:
            if self.video_player.is_finished:
                return True
            ok_ref, reference_frame = self.video_player.read()
            ok_cam, webcam_frame = self.camera.read()
            if not ok_ref or reference_frame is None:
                return True
            if not ok_cam or webcam_frame is None:
                return False
            if self.mirror_camera:
                webcam_frame = cv2.flip(webcam_frame, 1)

            self._update_fps()
            detector_timestamp = max(int(self.video_player.timestamp_ms), self._last_detector_ms + 1)
            self._last_detector_ms = detector_timestamp
            landmarks, _ = detect_pose_in_bgr(webcam_frame, self.pose_context, detector_timestamp)
            realtime_pose = create_pose_frame(frame_index, self.video_player.timestamp_ms, landmarks)
            result = self.engine.compare(realtime_pose)
            score = self.engine.current_score()
            view = self.hud.draw(
                reference_frame=reference_frame,
                webcam_frame=webcam_frame,
                realtime_pose=realtime_pose,
                result=result,
                fps=self.fps,
                score=score,
                feedback=result.feedback,
            )
            self.hud.show(view)
            frame_index += 1
            if self._should_quit():
                return False

    def _compute_final_result(self):
        duration_seconds = self.video_player.frame_count / self.video_player.fps
        return self.final_score_calculator.calculate(
            feedbacks=self.engine.feedback_history,
            similarity_average=self.engine.similarity_average(),
            duration_seconds=duration_seconds,
        )

    def _show_final_result(self, result):
        screen = self.hud.draw_final_result(result)
        while True:
            cv2.imshow("Final Score", screen)
            key = cv2.waitKey(50) & 0xFF
            if key in (ord("q"), ord("Q"), 27):
                break

    def _update_fps(self):
        current = time.perf_counter()
        elapsed = current - self._last_tick
        if elapsed > 0:
            current_fps = 1.0 / elapsed
            self.fps = current_fps if self.fps == 0.0 else (self.fps * 0.9) + (current_fps * 0.1)
        self._last_tick = current

    @staticmethod
    def _should_quit():
        key = cv2.waitKey(1) & 0xFF
        return key in (ord("q"), ord("Q"), 27)

    def _release_resources(self):
        if self.camera is not None:
            self.camera.release()
        if self.video_player is not None:
            self.video_player.release()
        cv2.destroyAllWindows()


RUN_SANDBOX_GAME = True
sandbox_game = DanceComparisonSandbox()

if RUN_SANDBOX_GAME:
    try:
        sandbox_result = sandbox_game.run()
        if sandbox_result is not None:
            print(f"Pontuação final: {sandbox_result.final_score:.1f}")
            print(f"Rank: {sandbox_result.ranque}")
    except Exception as exc:
        print(f"Sandbox não pôde ser executado: {exc}")
else:
    print("Sandbox pronto. Execute sandbox_game.run() para jogar a Macarena.")


Processando macareninha.mp4 para gerar macareninha_moveset.json e macareninha_pose.mp4...
Processados 50 frames...
Processados 100 frames...
Processados 150 frames...
Vídeo compatível com VS Code/notebook salvo em: macareninha_pose.mp4
Moveset salvo em: macareninha_moveset.json
Vídeo de debug salvo em: macareninha_pose.mp4
Original: 180 frames, 8.265 FPS, 21.779s
Processado: 180 frames, FPS efetivo 8.265
Landmarks visíveis por frame: média 18.1, mínimo 6, máximo 25
Pontuação final: 61.1
Rank: C


## 8. Conclusão

A partir dos resultados encontrados nesse notebook foi possível ter em mente que é realmente possível criar um jogo de dança utilizando a biblioteca do MediaPipe.

O projeto mostra como visão computacional pode transformar um vídeo comum em uma coreografia jogável.

O MediaPipe detecta o corpo, o sistema normaliza os landmarks e o algoritmo compara a pose do jogador com o moveset da dança.

A parte mais importante não é apenas detectar pontos do corpo, mas transformar esses pontos em informação útil: posição, escala, ângulos, movimento e participação.

No aplicativo Android, a implementação foi otimizada com pipeline nativo, buffers reutilizáveis e processamento em tempo real. Neste notebook, a mesma ideia foi simplificada para ficar mais fácil de entender e experimentar.

### Limitações do notebook

Este notebook prioriza clareza.

Por isso, ele não reproduz toda a arquitetura Android, nem as otimizações com CameraX, MediaCodec, libyuv e MPImage.

Mesmo assim, ele mantém o núcleo do sistema:

```text
detectar pose -> normalizar -> calcular ângulos -> comparar com moveset -> gerar score
```

Essa é a base conceitual usada pelo aplicativo.


## 9. Referências

- MediaPipe — Pose Landmarker: https://ai.google.dev/edge/mediapipe/solutions/vision/pose_landmarker
- MediaPipe — Python solutions: https://ai.google.dev/edge/mediapipe/solutions/guide
- OpenCV — VideoCapture: https://docs.opencv.org/4.x/d8/dfe/classcv_1_1VideoCapture.html
- OpenCV — VideoWriter: https://docs.opencv.org/4.x/dd/d9e/classcv_1_1VideoWriter.html
- Python — módulo `json`: https://docs.python.org/3/library/json.html
- Python — módulo `math`: https://docs.python.org/3/library/math.html

##### Algumas inspirações:
- A Real Time Dance Analysis Program to Assist in Dance Practice Using Pose Estimation: https://www.youtube.com/watch?v=bfS_ATn_rqg
- MEDIAPIPE POSE | Detecção de KEYPOINTS humanos com Visão Computacional: https://www.youtube.com/watch?v=Fcb6OEPi8kM
- UBISOFT PARIS. Just Dance 4. Montreuil: Ubisoft, 2012. Videogame.
